In [1]:
import os
import cv2
import numpy as np
import pandas as pd
from tqdm import tqdm
from PIL import Image
import matplotlib.pyplot as plt

import timm
import torch
import torch.nn as nn
import torchvision
import torch.nn.functional as F
import torchvision.transforms as T
from torch.cuda.amp import autocast, GradScaler
from torch.utils.data import random_split, Dataset, DataLoader

from transformers import DistilBertModel, DistilBertConfig, DistilBertTokenizer

## Configuration

In [2]:
class CFG:
    data_path = "/kaggle/input/datasets/hsankesara/flickr-image-dataset/flickr30k_images/flickr30k_images"
    caption_path = "/kaggle/input/datasets/hsankesara/flickr-image-dataset/flickr30k_images/results.csv"

    imageModel = 'convnext_small'
    textModel = "distilbert-base-uncased"
    textTokenizer = 'distilbert-base-uncased'
    pre_trained = True
    trainable = True

    batch_size=32
    max_length=77

    epochs = 5

    img_embed_size, text_embed_size = 768, 768

    proj_embed_size = 512

## Data Loading and pre processing

In [3]:
transforms = T.Compose([
    T.Resize((224, 224)),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

In [4]:
class flickrDataset(Dataset):
    def __init__(self, data_path=CFG.data_path, caption_path=CFG.caption_path, model_name=CFG.textTokenizer, transforms=None):
        self.data_path = data_path
        self.transforms = transforms
        
        self.tokenizer = DistilBertTokenizer.from_pretrained(model_name)
        self.df = pd.read_csv(caption_path, delimiter='|')
        self.df = self.df.dropna(subset=[' comment']).reset_index(drop=True)

        unique_images = self.df['image_name'].unique()
        self.img_name_to_id = {name: idx for idx, name in enumerate(unique_images)}
        self.num_unique_images = len(unique_images)

    def __len__(self):
        return self.df.shape[0]

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image_path = os.path.join(self.data_path, row['image_name'])
        img = Image.open(image_path).convert("RGB")
        cap = row[' comment']
        image_id = self.img_name_to_id[row['image_name']]

        if self.transforms:
            img = self.transforms(img)
        txt = self.tokenizer(cap, padding='max_length', truncation=True, max_length=CFG.max_length, return_tensors="pt")

        txt = {k: v.squeeze(0) for k, v in txt.items()}
        
        return img, txt, image_id

In [5]:
dataset = flickrDataset(transforms=transforms)

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [6]:
val_ratio = 0.2
val_size = int(len(dataset) * val_ratio)
train_size = len(dataset) - val_size

train_dataset, val_dataset = random_split(dataset, [train_size, val_size])

In [7]:
train_loader = DataLoader(train_dataset, batch_size=CFG.batch_size, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=CFG.batch_size, shuffle=False, num_workers=2)

In [8]:
for img, txt, ids in train_loader:
    print(type(img), img.shape, end="**")
    print(txt.keys())
    break

<class 'torch.Tensor'> torch.Size([32, 3, 224, 224])**dict_keys(['input_ids', 'attention_mask'])


## Image Encoder

In [9]:
class ImageEncoder(nn.Module):
    def __init__(self, model_name=CFG.imageModel, pre_trained=CFG.pre_trained, trainable=CFG.trainable):
        super(ImageEncoder, self).__init__()
        self.model = timm.create_model(model_name, pretrained=pre_trained, num_classes=0, global_pool='avg')

        for p in self.model.parameters():
            p.requires_grad=trainable

    def forward(self, x):
        return self.model(x)

## Text Enocder

In [10]:
class TextEncoder(nn.Module):
    def __init__(self, model_name=CFG.textModel, pre_trained=CFG.pre_trained, trainable=CFG.trainable):
        super(TextEncoder, self).__init__()

        if pre_trained:
            self.model = DistilBertModel.from_pretrained(model_name)
        else:
            self.model = DistilBertModel(config=DistilBertConfig())
            
        for p in self.model.parameters():
            p.requires_grad = trainable

    def forward(self, x):
        out = self.model(input_ids=x['input_ids'], attention_mask=x['attention_mask'])
        mask = x['attention_mask'].unsqueeze(-1)  # (B, seq_len, 1)
        return (out.last_hidden_state * mask).sum(dim=1) / mask.sum(dim=1)

## CLIP

In [11]:
class MiniCLIP(nn.Module):
    def __init__(self):
        super(MiniCLIP, self).__init__()
        self.imageEncoder = ImageEncoder()
        self.textEncoder = TextEncoder()

        self.img_proj = nn.Linear(CFG.img_embed_size, CFG.proj_embed_size)
        self.text_proj = nn.Linear(CFG.text_embed_size, CFG.proj_embed_size)
        
        self.temp = nn.Parameter(torch.tensor(0.07))

    def encode_image(self, x):
        img_embeddings = self.imageEncoder(x)
        img_embeddings = self.img_proj(img_embeddings)
        return F.normalize(img_embeddings, p=2, dim=-1)

    def encode_text(self, y):
        text_embeddings = self.textEncoder(y)
        text_embeddings = self.text_proj(text_embeddings)
        return F.normalize(text_embeddings, p=2, dim=-1)
    
    def forward(self, x, y):
        img_embeddings = self.encode_image(x)
        text_embeddings = self.encode_text(y)
    
        logits = img_embeddings @ text_embeddings.T 
        logits = logits / self.temp
    
        return logits

In [12]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [13]:
clip_model = MiniCLIP().to(device)

model.safetensors:   0%|          | 0.00/201M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_transform.weight  | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [14]:
optimizer = torch.optim.Adam(clip_model.parameters(), lr=1e-4)

In [15]:
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=CFG.epochs * len(train_loader))

In [16]:
criterion = nn.CrossEntropyLoss()

In [17]:
total = sum(p.numel() for p in clip_model.parameters() if p.requires_grad)
print(f'Trainable params: {total:,}')

Trainable params: 116,605,025


## Training

In [18]:
num_epochs = CFG.epochs
scaler = GradScaler()

for epoch in range(num_epochs):
    # Trianing
    clip_model.train()
    train_loss = 0.0

    loop = tqdm(train_loader, desc=f"Epoch [{epoch+1}/{num_epochs}] Train", leave=False)
    for batch_idx, (images, text, ids) in enumerate(loop):
        images = images.to(device)
        text = {k: v.to(device) for k, v in text.items()}

        optimizer.zero_grad()

        with autocast():
            logits = clip_model(images, text)
            batch_size = images.size(0)
            labels = torch.arange(batch_size).to(device)
            loss_i = criterion(logits, labels)
            loss_t = criterion(logits.T, labels)
            loss = (loss_i + loss_t) / 2

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()

        train_loss += loss.item() * batch_size
        if batch_idx % 200 == 0:   
            loop.set_postfix(loss=loss.item(), lr=scheduler.get_last_lr()[0])

    avg_train = train_loss / len(train_dataset)

    #  Validation
    clip_model.eval()
    val_loss = 0.0

    with torch.no_grad():
        for images, text, ids in tqdm(val_loader, desc=f"Epoch [{epoch+1}/{num_epochs}] Val", leave=False):
            images = images.to(device)
            text = {k: v.to(device) for k, v in text.items()}

            with autocast():
                logits = clip_model(images, text)
                batch_size = images.size(0)
                labels = torch.arange(batch_size).to(device)
                loss_i = criterion(logits, labels)
                loss_t = criterion(logits.T, labels)
                loss = (loss_i + loss_t) / 2

            val_loss += loss.item() * batch_size

    avg_val = val_loss / len(val_dataset)

    print(f"Epoch [{epoch+1}/{num_epochs}]  Train Loss: {avg_train:.4f}  Val Loss: {avg_val:.4f}  LR: {scheduler.get_last_lr()[0]:.6f}")

/tmp/ipykernel_24/705852154.py:2: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()
Epoch [1/5] Train:   0%|          | 0/3973 [00:00<?, ?it/s]/tmp/ipykernel_24/705852154.py:16: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
/tmp/ipykernel_24/705852154.py:27: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  scheduler.step()
Epoch [1/5] Val:   0%|          | 0/994 [00:00<?, ?it/s]/tmp/ipykernel_24/705852154.py:44: FutureWarning: `torch.cuda.amp.autocast(args

Epoch [1/5]  Train Loss: 0.6022  Val Loss: 0.3829  LR: 0.000090


Epoch [2/5]  Train Loss: 0.2465  Val Loss: 0.2672  LR: 0.000065


Epoch [3/5]  Train Loss: 0.1249  Val Loss: 0.2077  LR: 0.000035


Epoch [4/5]  Train Loss: 0.0574  Val Loss: 0.1713  LR: 0.000010


Epoch [5/5]  Train Loss: 0.0322  Val Loss: 0.1647  LR: 0.000000


In [19]:
torch.save(clip_model.state_dict(), 'model_weights.pth')

## Evaulation

In [20]:
@torch.no_grad()
def extract_embeddings(model, dataloader, device):

    clip_model.eval()
    all_img_embeds = []
    all_txt_embeds = []
    all_image_ids = []

    for images, texts, image_ids in tqdm(dataloader, desc="Extracting embeddings"):
        images = images.to(device)
        texts = {k: v.to(device) for k, v in texts.items()}

        img_emb = clip_model.encode_image(images)
        txt_emb = clip_model.encode_text(texts)

        all_img_embeds.append(img_emb.cpu())
        all_txt_embeds.append(txt_emb.cpu())
        all_image_ids.append(image_ids)

    img_embeds = torch.cat(all_img_embeds, dim=0)   # (N, proj_dim)
    txt_embeds = torch.cat(all_txt_embeds, dim=0)   # (N, proj_dim)
    image_ids  = torch.cat(all_image_ids, dim=0)    # (N,)

    return img_embeds, txt_embeds, image_ids

In [21]:
def compute_recall_at_k(similarity, ground_truth, k_values=[1, 5, 10]):
    recalls = {}
    num_queries = similarity.shape[0]

    for k in k_values:
        top_k_indices = similarity.topk(k, dim=1).indices  # (num_queries, k)
        hits = 0
        for i in range(num_queries):
            retrieved = set(top_k_indices[i].tolist())
            if retrieved & ground_truth[i]:  # intersection
                hits += 1
        recalls[f"R@{k}"] = hits / num_queries

    return recalls

In [22]:
def evaluate_recall(img_embeds, txt_embeds, image_ids, k_values=[1, 5, 10]):
    
    N = len(image_ids)

    from collections import defaultdict
    img_id_to_caption_indices = defaultdict(set)
    for cap_idx in range(N):
        img_id = image_ids[cap_idx].item()
        img_id_to_caption_indices[img_id].add(cap_idx)

    seen_images = {}
    i2t_query_indices = []
    i2t_ground_truth = {}

    for cap_idx in range(N):
        img_id = image_ids[cap_idx].item()
        if img_id not in seen_images:
            seen_images[img_id] = len(i2t_query_indices)
            i2t_query_indices.append(cap_idx)
            i2t_ground_truth[seen_images[img_id]] = img_id_to_caption_indices[img_id]

    t2i_ground_truth = {}
    for cap_idx in range(N):
        img_id = image_ids[cap_idx].item()
        t2i_ground_truth[cap_idx] = {seen_images[img_id]}

    
    unique_img_embeds = img_embeds[i2t_query_indices]  
    sim_i2t = unique_img_embeds @ txt_embeds.T         

    
    sim_t2i = txt_embeds @ unique_img_embeds.T          

    # Compute Recall
    i2t_recall = compute_recall_at_k(sim_i2t, i2t_ground_truth, k_values)
    t2i_recall = compute_recall_at_k(sim_t2i, t2i_ground_truth, k_values)

    return i2t_recall, t2i_recall

In [23]:
img_embeds, txt_embeds, image_ids = extract_embeddings(clip_model, val_loader, device)

print(f"Image embeddings: {img_embeds.shape}")
print(f"Text embeddings:  {txt_embeds.shape}")
print(f"Unique images:    {image_ids.unique().shape[0]}")
print()

# Compute Recall@K
i2t_recall, t2i_recall = evaluate_recall(img_embeds, txt_embeds, image_ids)

print("=" * 45)
print("  Image → Text Retrieval (I2T)")
print("=" * 45)
for k, v in i2t_recall.items():
    print(f"  {k}: {v:.4f}  ({v*100:.2f}%)")

print()
print("=" * 45)
print("  Text → Image Retrieval (T2I)")
print("=" * 45)
for k, v in t2i_recall.items():
    print(f"  {k}: {v:.4f}  ({v*100:.2f}%)")

print()
print("=" * 45)
print(f"  Mean R@1:  {(i2t_recall['R@1'] + t2i_recall['R@1']) / 2:.4f}")
print(f"  Mean R@5:  {(i2t_recall['R@5'] + t2i_recall['R@5']) / 2:.4f}")
print(f"  Mean R@10: {(i2t_recall['R@10'] + t2i_recall['R@10']) / 2:.4f}")
print("=" * 45)

Extracting embeddings: 100%|██████████| 994/994 [05:10<00:00,  3.20it/s]


Image embeddings: torch.Size([31782, 512])
Text embeddings:  torch.Size([31782, 512])
Unique images:    21457

  Image → Text Retrieval (I2T)
  R@1: 0.2769  (27.69%)
  R@5: 0.5399  (53.99%)
  R@10: 0.6585  (65.85%)

  Text → Image Retrieval (T2I)
  R@1: 0.2524  (25.24%)
  R@5: 0.5108  (51.08%)
  R@10: 0.6313  (63.13%)

  Mean R@1:  0.2647
  Mean R@5:  0.5253
  Mean R@10: 0.6449
